<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/kinematics/04_numerical_ik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K4 ヤコビ行列と数値逆運動学(notebook)

本文:[`04-numerical-ik.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/kinematics/04-numerical-ik.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

l1, l2 = 1.0, 0.8                                           # リンクの長さ [m](K1・K2 と同じ)

def fk(q, l1=l1, l2=l2):
    # 順運動学(K1):手先の位置 (x, y)
    q1, q2 = q
    return np.array([l1 * np.cos(q1) + l2 * np.cos(q1 + q2), l1 * np.sin(q1) + l2 * np.sin(q1 + q2)])

def ik_analytic(p, sign=+1, l1=l1, l2=l2):
    # 解析的な逆運動学(K2)。sign=+1 なら q2 > 0 の解。届かなければ None
    x, y = p
    c = (x**2 + y**2 - l1**2 - l2**2) / (2 * l1 * l2)
    if abs(c) > 1:
        return None
    q2 = sign * np.arccos(c)
    return np.array([np.arctan2(y, x) - np.arctan2(l2 * np.sin(q2), l1 + l2 * np.cos(q2)), q2])

def draw_arm(ax, q, **kw):
    elbow = np.array([l1 * np.cos(q[0]), l1 * np.sin(q[0])])
    pts = np.array([[0, 0], elbow, fk(q)])
    ax.plot(pts[:, 0], pts[:, 1], "o-", **kw)

## 1. ヤコビ行列

本文の式 (2) を実装し、数値微分で確かめる。

**手計算**:本文の確認課題1・2を解く。

- 手計算:(ここに書く)

In [ ]:
def jacobian(q, l1=l1, l2=l2):
    q1, q2 = q
    raise NotImplementedError("TODO: 本文の式 (2) の 2x2 の行列を np.array で返す")

In [ ]:
def jacobian_numeric(q, h=1e-6):
    J = np.zeros((2, 2))
    for j in range(2):
        dq = np.zeros(2); dq[j] = h
        J[:, j] = (fk(q + dq) - fk(q - dq)) / (2 * h)
    return J

check("確認課題1:q = (0, pi/2)", jacobian([0.0, np.pi / 2]), [[-0.8, -0.8], [1.0, 0.0]])
for q in [np.array([0.7, -0.4]), np.array([2.0, 1.3]), np.random.default_rng(0).uniform(-np.pi, np.pi, 2)]:
    check(f"q = {q.round(2)}:式 (2) = 数値微分", jacobian(q), jacobian_numeric(q), tol=1e-8)
    check(f"q = {q.round(2)}:1列目は (-y, x)", jacobian(q)[:, 0], [-fk(q)[1], fk(q)[0]])

## 2. 特異姿勢

本文の式 (3):$\det J = l_1 l_2 \sin q_2$ を確かめ、 $q_2$ を 0 に近づけたときの様子を見る。 $J$ の2つの特異値(M8)のうち小さいほうは、「最も動きにくい方向に、関節を単位の速さで回したときの手先の速さ」である。

In [ ]:
for q2 in [np.pi / 2, 0.5, 0.1, 0.01, 0.0]:
    J = jacobian([0.3, q2])
    sv = np.linalg.svd(J, compute_uv=False)
    print(f"q2 = {q2:6.3f}:det J = {np.linalg.det(J):7.4f}  (l1 l2 sin q2 = {l1 * l2 * np.sin(q2):7.4f})  特異値 {sv.round(4)}")
check("q2 = 0 では det J = 0", np.linalg.det(jacobian([0.3, 0.0])), 0.0, tol=1e-12)
J0 = jacobian([0.0, 0.0])
print("確認課題2:q = (0, 0) の J の列:", J0[:, 0], J0[:, 1], " → どちらも y の向き。x の向き(腕を伸ばす向き)には動けない")

## 3. 数値逆運動学

本文 §3 の手順を実装する。`ik_step` は、式 (4) を `np.linalg.solve` で解いて、更新した $q$ を返す。

**予想**:誤差は、くり返しのたびにどう小さくなるか(半分ずつ? もっと速く?)。

- 予想:(ここに書く)

In [ ]:
def ik_step(q, p_target):
    raise NotImplementedError("TODO: e = p_target - fk(q)、J dq = e を solve で解き、q + dq を返す")

def numerical_ik(q0, p_target, step=None, tol=1e-10, max_iters=50):
    step = step or ik_step
    q = np.array(q0, dtype=float)
    errors = [np.linalg.norm(p_target - fk(q))]
    for _ in range(max_iters):
        if errors[-1] < tol:
            break
        q = step(q, p_target)
        errors.append(np.linalg.norm(p_target - fk(q)))
    return q, np.array(errors)

In [ ]:
target = np.array([1.2, 0.6])
q_sol, errs = numerical_ik([0.3, 0.5], target)
print("誤差の変化:", [f"{e:.1e}" for e in errs])
check("目標に着いた", fk(q_sol), target, tol=1e-9)
check("q2 > 0 の解析解と同じ解に着いた", q_sol, ik_analytic(target, +1), tol=1e-8)
q_sol2, _ = numerical_ik([1.0, -0.5], target)
check("初めの値を変えると、もう一方の解(q2 < 0)に着く", q_sol2, ik_analytic(target, -1), tol=1e-8)
fig, ax = plt.subplots(figsize=(4, 4))
draw_arm(ax, q_sol, label="from q0=(0.3, 0.5)"); draw_arm(ax, q_sol2, label="from q0=(1.0, -0.5)")
ax.plot(*target, "r*", ms=12); ax.set_aspect("equal"); ax.legend(fontsize=8); ax.grid(); plt.show()

## 4. 届かない目標と、特異姿勢の近く

届かない目標 $(3, 0)$ で、式 (4) をそのまま解く方法と、疑似逆行列、減衰最小二乗法(本文の式 (5))を比べる。

**実装**:減衰最小二乗法の1ステップ `ik_step_dls` を書く。

In [ ]:
def ik_step_pinv(q, p_target):
    return q + np.linalg.pinv(jacobian(q)) @ (p_target - fk(q))

def ik_step_dls(q, p_target, lam=0.5):
    J, e = jacobian(q), p_target - fk(q)
    raise NotImplementedError("TODO: dq = (J^T J + lam^2 I)^{-1} J^T e を solve で解き、q + dq を返す")

In [ ]:
check("lam = 0 なら、減衰最小二乗法 = 式 (4)", ik_step_dls(np.array([0.3, 0.5]), target, lam=0.0), ik_step(np.array([0.3, 0.5]), target))
far = np.array([3.0, 0.0])
for name, step in [("solve", ik_step), ("pinv", ik_step_pinv), ("DLS lam=0.5", ik_step_dls)]:
    q = np.array([0.3, 0.5]); dq_max = 0.0
    with np.errstate(all="ignore"):
        for _ in range(30):
            q_new = step(q, far)
            dq_max = max(dq_max, np.abs(q_new - q).max()); q = q_new
    print(f"{name:12s}:30回後の手先 {fk(q).round(3)}  q2 = {q[1]:9.3f}  1回の関節の動きの最大 {dq_max:9.2f} rad")

届かない目標に対しては、腕を目標の方向にまっすぐ伸ばした姿勢(手先 $(1.8, 0)$ 、 $q = (0, 0)$)が、いちばん近い答えである。そこは特異姿勢( $q_2 = 0$)なので、近づくほど $J$ が逆行列を持たなくなる。式 (4) をそのまま解く方法と疑似逆行列では、関節が1回に何十ラジアンも回り、手先はあちこちに飛ぶ。減衰最小二乗法では、動きが抑えられ、腕を伸ばした姿勢に落ち着く。

## 5. 条件を変える

1. 減衰の強さ $\lambda$ を 0.01、0.1、0.5、1.0 に変えて、届く目標 $(1.2, 0.6)$ に着くまでの回数と、届かない目標 $(3, 0)$ での関節の動きの最大を比べる
2. 初めの値を $q_2 = 0$(特異姿勢)にすると、式 (4) をそのまま解く方法は何が起きるか

In [ ]:
for lam in [0.01, 0.1, 0.5, 1.0]:
    _, e1 = numerical_ik([0.3, 0.5], target, step=lambda q, p: ik_step_dls(q, p, lam), max_iters=500)
    q = np.array([0.3, 0.5]); dq_max = 0.0
    for _ in range(30):
        q_new = ik_step_dls(q, far, lam); dq_max = max(dq_max, np.abs(q_new - q).max()); q = q_new
    print(f"1. lam = {lam:4}:届く目標に着くまで {len(e1) - 1:3d} 回、届かない目標で 30 回後の手先 {fk(q).round(3)}、関節の動きの最大 {dq_max:.3f} rad")
try:
    ik_step(np.array([0.3, 0.0]), target)
    print("2. 解けた(J がぎりぎり特異でない数値になった)")
except np.linalg.LinAlgError as err:
    print("2. q2 = 0 から始めると solve が失敗する:", err)
print("   減衰最小二乗法なら進める:", numerical_ik([0.3, 0.0], target, step=ik_step_dls, max_iters=200)[1][-1].round(12))

## 6. 円の軌道に沿って動かす

手先に円を描かせる(本文 §5)。1つ前の点の解を初めの値にして、各点で数回くり返す。円が特異姿勢の近く(腕がほぼ伸びきる所)を通るようにすると、関節の動きはどうなるか。

**予想**:半径を大きくして、円の一部が到達可能領域の外側の境界に近づくと、関節の速さ(1点あたりの関節角の変化)はどうなるか。

- 予想:(ここに書く)

In [ ]:
def track_circle(center, radius, n=200, iters=5, step=ik_step_dls):
    ts = np.linspace(0, 2 * np.pi, n)
    path = np.column_stack([center[0] + radius * np.cos(ts), center[1] + radius * np.sin(ts)])
    q = ik_analytic(path[0], +1)
    qs, errs = [q], []
    for p in path[1:]:
        for _ in range(iters):
            q = step(q, p)
        qs.append(q); errs.append(np.linalg.norm(fk(q) - p))
    return path, np.array(qs), np.array(errs)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for radius, color in [(0.4, "b"), (0.75, "r")]:
    path, qs, errs = track_circle((1.0, 0.0), radius)
    dq = np.abs(np.diff(qs, axis=0)).max(axis=1)
    hand = np.array([fk(q) for q in qs])
    ax[0].plot(path[:, 0], path[:, 1], color + "--", lw=1); ax[0].plot(hand[:, 0], hand[:, 1], color, label=f"r={radius}")
    ax[1].plot(dq, color, label=f"r={radius}")
    print(f"半径 {radius}:追従の誤差の最大 {errs.max():.1e} m、1点あたりの関節の動きの最大 {dq.max():.3f} rad、|sin q2| の最小 {np.abs(np.sin(qs[:, 1])).min():.3f}")
ax[0].add_patch(plt.Circle((0, 0), l1 + l2, fill=False, color="gray")); ax[0].set_aspect("equal"); ax[0].legend(); ax[0].set_title("hand path")
ax[1].set_xlabel("point"); ax[1].set_ylabel("max |dq| per point [rad]"); ax[1].legend()
plt.tight_layout(); plt.show()

半径を大きくすると、円が外側の境界(腕が伸びきる所)に近づき、 $\lvert \sin q_2 \rvert$ が小さくなる。そこでは関節の動きが大きくなり、減衰最小二乗法の減衰が効いて、追従の誤差も大きくなる(動きを抑える代わりに、目標に着ききらない)。減衰の強さを変えて確かめてみる。

## 7. 研究で使うときは

`cpai_lab` には運動学の関数は入っていない(本文 §7)。シミュレータや ROS 2 の運動学のライブラリを使うときも、この notebook と同じ点を確かめる。

- 逆運動学が**収束したか**(誤差が十分小さいか、回数の上限で止まったのか)
- **どちらの解**に着いたか(初めの値で変わる)
- **特異姿勢の近く**を通っていないか( $\det J$ や、小さいほうの特異値を記録する)

## 8. 説明(自分の言葉で書く)

1. ヤコビ行列の各列は何を表しているか
   - (ここに書く)
2. 数値逆運動学で、一次近似の計算をくり返すと目標に着くのはなぜか。初めの値で着く解が変わるのはなぜか
   - (ここに書く)
3. 特異姿勢の近くで、関節が大きく動くのはなぜか。減衰最小二乗法は何をしているか
   - (ここに書く)